# 04 Outcomes and metrics

Turns every method's predictions into applied corrections, measures them against the reference (the labelled slots) in energy and in days, pools the numbers with station-bootstrap intervals, checks the safety gate, and runs the operating-point study of the control c.

Abbreviations used here: **RPF** is reverse power flow, the condition where a distribution substation exports power because rooftop solar exceeds local demand; a *wrong RPF sign* is a meter recording that stores the export as an import. **M7** is the deterministic threshold rule, **M8** the two-stage XGBoost classifier and **M9** the counterfactual bridge method of the `pynrpf` package. **MW** and **MWh** are megawatts and megawatt-hours; one interval is 15 minutes, and a *slot* counts intervals from midnight (slot 24 is 06:00).

**Inputs.** `results/02_baselines/intervals_m7.parquet` and `intervals_m8.parquet`, `results/03_m9/`, `results/01_data_folds/`.

**Outputs.** `results/04_metrics/`: `site_days.parquet` and `site_day_decisions.csv` (one row per method and site-day with the energy terms, window agreement and minimum-demand change), `pooled.csv`, `stations.csv`, `macro.csv`, `bootstrap.csv`, `coverage.csv`, `sensitivity.csv`, `calibration_reliability.csv`, `gate.json`, and `operating_points/` (targets, frontier, heatmaps, selections and their figures); `results/manifests/04_metrics.json`.

**Runtime.** A few minutes; the bootstrap draws 1,000 station resamples.

**Steps.**

1. Setup.
2. Reference terms, metrics and the operating-point study.
3. Read the headline table and the gate.

## 1. Setup

Locate the article folder, import the paper code and load the configuration. Loading the configuration verifies the SHA-256 of every dataset, so a wrong or edited data file stops the run here. `CONFIG` is the one knob: point it at another YAML to run a variant into another folder.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display


def article_root() -> Path:
    """publication/2_journal_article, found from this folder, JupyterLab's root or the repository root."""
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "paper" / "stages.py").exists():
            return candidate
        nested = candidate / "publication" / "2_journal_article"
        if (nested / "paper" / "stages.py").exists():
            return nested
    raise FileNotFoundError("Could not locate publication/2_journal_article.")


ARTICLE = article_root()
sys.path.insert(0, str(ARTICLE))
from paper import config, results, stages  # noqa: E402

CONFIG = ARTICLE / "config" / "evaluation.yaml"   # point this at another configuration to run a variant
SETTINGS = config.load(CONFIG)                     # verifies the dataset hashes before anything runs
RESULTS = SETTINGS.output_root()
print("results folder:", RESULTS.relative_to(ARTICLE))

## 2. Metrics

The correction energy of a slot is 2·y·0.25 MWh. Energy IoU is the overlap of the flipped and the reference energy over their union; energy precision is the share of the flipped energy the reference supports, gated at 0.90 on Beta sure days. Site-day precision, recall and F1 count days. Unsure Beta days never enter the headline numbers; they appear in the sensitivity table only.

In [ ]:
out = stages.metrics(SETTINGS)

## 3. The headline and the gate

One row per method and group (both datasets, Alpha, Beta sure). The gate decision names the default method.

In [ ]:
pooled = results.metric(SETTINGS, 'pooled')
display(pooled.set_index(['method', 'group'])[['n_days', 'n_rpf', 'energy_iou', 'energy_precision', 'day_f1', 'day_precision', 'sure_day_recall']].round(3))
results.gate(SETTINGS)

## Result

Every metric the paper reports is on disk. Notebook 05 adds the forecasting case study; `paper_figures.ipynb` and `paper_tables.ipynb` draw the artefacts from these files.